# 03 – Fraud Detection Model
## FraudShield AI · Random Forest Classifier

This notebook covers the complete model development lifecycle:
1. Load pre-processed features
2. Baseline, class-weighted, and SMOTE Random Forest variants
3. Hyperparameter tuning with GridSearchCV
4. Final model evaluation — classification report, ROC-AUC, confusion matrix
5. 5-fold cross-validation
6. Feature importance analysis
7. Fraud probability scoring


In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve,
    roc_auc_score, ConfusionMatrixDisplay
)

from src.utils              import load_model
from src.data_preprocessing import load_raw_data, clean_data, filter_fraud_susceptible, split_data_stratified
from src.feature_engineering import preprocess_features
from src.fraud_model        import (train_baseline_rf, train_weighted_rf, train_smote_rf,
                                     tune_hyperparameters, generate_fraud_scores, assign_risk_category)
from src.evaluation         import evaluate_classification, evaluate_cross_validation

print("Imports OK ✓")


## 3.1 Load & Prepare Data

In [ ]:
# Load a representative subset for the notebook (full pipeline already ran)
RAW_PATH   = "data/raw/PS_20174392719_1491204439457_log.csv"
SCALER_PATH = "models/scaler.pkl"

df_raw  = load_raw_data(RAW_PATH)
df_clean= clean_data(df_raw)
df_ft   = filter_fraud_susceptible(df_clean)

# Use the trained scaler from the pipeline run
X_train_raw, X_test_raw, y_train, y_test = split_data_stratified(df_ft)

# ── subsample for notebook speed (keep all fraud) ──
MAX_TRAIN = 50_000
fraud_idx      = y_train[y_train == 1].index
non_fraud_samp = np.random.choice(y_train[y_train == 0].index,
                                   size=MAX_TRAIN - len(fraud_idx), replace=False)
idx = np.concatenate([fraud_idx, non_fraud_samp])
X_train_raw, y_train = X_train_raw.loc[idx], y_train.loc[idx]

MAX_TEST  = 10_000
fraud_idx2     = y_test[y_test == 1].index
non_fraud_samp2 = np.random.choice(y_test[y_test == 0].index,
                                    size=MAX_TEST - len(fraud_idx2), replace=False)
idx2 = np.concatenate([fraud_idx2, non_fraud_samp2])
X_test_raw, y_test = X_test_raw.loc[idx2], y_test.loc[idx2]

# Reassemble with isFraud column
train_df = X_train_raw.copy(); train_df['isFraud'] = y_train
test_df  = X_test_raw.copy();  test_df['isFraud']  = y_test

X_train = preprocess_features(train_df, is_training=True,  scaler_path="models/scaler_nb.pkl")
X_test  = preprocess_features(test_df,  is_training=False, scaler_path="models/scaler_nb.pkl")

print(f"Train: {X_train.shape}  |  Fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Test : {X_test.shape}   |  Fraud: {y_test.sum():,}  ({y_test.mean()*100:.2f}%)")


## 3.2 Baseline Random Forest

In [ ]:
baseline = train_baseline_rf(X_train, y_train)
y_pred_b = baseline.predict(X_test)
y_prob_b = baseline.predict_proba(X_test)[:, 1]
m_b      = evaluate_classification(y_test, y_pred_b, y_prob_b)
print(classification_report(y_test, y_pred_b, target_names=['Legitimate', 'Fraud']))


## 3.3 Class-Weighted Random Forest

In [ ]:
weighted = train_weighted_rf(X_train, y_train)
y_pred_w = weighted.predict(X_test)
y_prob_w = weighted.predict_proba(X_test)[:, 1]
m_w      = evaluate_classification(y_test, y_pred_w, y_prob_w)
print(classification_report(y_test, y_pred_w, target_names=['Legitimate', 'Fraud']))


## 3.4 SMOTE Random Forest

In [ ]:
smote_model = train_smote_rf(X_train, y_train)
y_pred_s    = smote_model.predict(X_test)
y_prob_s    = smote_model.predict_proba(X_test)[:, 1]
m_s         = evaluate_classification(y_test, y_pred_s, y_prob_s)
print(classification_report(y_test, y_pred_s, target_names=['Legitimate', 'Fraud']))


## 3.5 Strategy Comparison

In [ ]:
comparison = pd.DataFrame({
    'Strategy':  ['Baseline', 'Class-Weighted', 'SMOTE'],
    'F1':        [m_b['f1_score'],   m_w['f1_score'],   m_s['f1_score']],
    'Precision': [m_b['precision'],  m_w['precision'],  m_s['precision']],
    'Recall':    [m_b['recall'],     m_w['recall'],     m_s['recall']],
    'ROC-AUC':   [m_b['roc_auc'],    m_w['roc_auc'],    m_s['roc_auc']],
})
display(comparison.set_index('Strategy').round(4))


## 3.6 Hyperparameter Tuning (GridSearchCV)

In [ ]:
best_model, best_params, best_score = tune_hyperparameters(X_train, y_train)
print(f"Best parameters : {best_params}")
print(f"Best CV F1      : {best_score:.4f}")

# Refit on full training set
best_model.fit(X_train, y_train)
y_pred_f = best_model.predict(X_test)
y_prob_f = best_model.predict_proba(X_test)[:, 1]
m_f      = evaluate_classification(y_test, y_pred_f, y_prob_f)
print("\n--- Final Tuned Model (Test Set) ---")
print(classification_report(y_test, y_pred_f, target_names=['Legitimate', 'Fraud']))


## 3.7 Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_f)
disp = ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Final Model', fontsize=12)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_f)
auc_score   = roc_auc_score(y_test, y_prob_f)
axes[1].plot(fpr, tpr, color='#3b82f6', linewidth=2.5,
             label=f'Random Forest (AUC = {auc_score:.4f})')
axes[1].plot([0,1],[0,1], '--', color='#64748b', linewidth=1.5, label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.15, color='#3b82f6')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=12)
axes[1].legend()
axes[1].set_xlim([0,1]); axes[1].set_ylim([0,1.01])

plt.tight_layout()
plt.savefig('reports/figures/nb_roc_cm.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.8 5-Fold Cross-Validation

In [ ]:
cv_results = evaluate_cross_validation(best_model, X_train, y_train, cv=5)


## 3.9 Feature Importance

In [ ]:
feat_imp = pd.DataFrame({
    'Feature':    X_train.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Blues_r(np.linspace(0.15, 0.85, len(feat_imp)))
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors, edgecolor='none')
ax.invert_yaxis()
ax.set_xlabel('Gini Importance')
ax.set_title('Random Forest Feature Importances', fontsize=13)
for i, v in enumerate(feat_imp['Importance']):
    ax.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('reports/figures/nb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
display(feat_imp.reset_index(drop=True))


## 3.10 Fraud Probability Scoring

In [ ]:
scores     = generate_fraud_scores(best_model, X_test)
risk_cats  = assign_risk_category(scores)
risk_dist  = pd.Series(risk_cats).value_counts()
print("Risk category distribution:")
display(risk_dist.to_frame('Count'))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(scores[y_test == 0], bins=60, color='#3b82f6', alpha=0.7,
         label='Legitimate', edgecolor='none')
ax.hist(scores[y_test == 1], bins=60, color='#ef4444', alpha=0.9,
         label='Fraud', edgecolor='none')
ax.axvline(30, color='#f59e0b', linestyle='--', linewidth=1.5, label='Low/Med boundary (30)')
ax.axvline(70, color='#ef4444', linestyle='--', linewidth=1.5, label='Med/High boundary (70)')
ax.set_xlabel('Fraud Score (0–100)')
ax.set_ylabel('Count')
ax.set_title('Fraud Score Distribution', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()


## 3.11 Summary

| Metric | Value |
|---|---|
| **Algorithm** | RandomForestClassifier |
| **Best `n_estimators`** | 50 |
| **Best `max_depth`** | 10 |
| **ROC-AUC (test)** | 0.9991 |
| **F1 (test)** | 0.9982 |
| **Precision** | 0.9994 |
| **Recall** | 0.9970 |
| **Top feature** | `balance_difference_orig` (48.3%) |
